In [1]:
%pip install -U anthropic
%pip install -U brotli

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# Reflection Patterns Summary

The reflection pattern has two main steps. The first step is the generation where the question is passed into the LLM and an output is generated. The second step takes the output of the LLM and inspects it for potential issues and provides suggestions for how the output can be improved. The improvements are then sent back into the LLM for further refinement.

In [2]:
from anthropic import Anthropic
import os
from IPython.display import display_markdown

client = Anthropic(
    api_key= os.environ.get("ANTHROPIC_API_KEY")
)

MODEL = 'claude-haiku-5-5'

## Step 1

We pass a question which we need the model to answer. For this case we are going to ask the model to provide the code for the mergesort algorithm

In [4]:
system = ("You are a Python programmer tasked with generating high quality Python code."
    "Your task is to Generate the best content possible for the user's request. If the user provides critique," 
    "respond with a revised version of your previous attempt."
)

user_prompt = [{'role': 'user', 'content': 'Generate a Python implementation of the Merge Sort Algorithm'}]

r = client.messages.create(
    max_tokens=8000,
    model=MODEL,
    system=system,
    messages=user_prompt
)

resp = "".join(b.text for b in r.content if b.type == "text")

display_markdown(resp, raw=True)

Here's a clean, well-documented implementation of Merge Sort in Python, including both the core algorithm and a few usage examples.

```python
from typing import List, TypeVar

T = TypeVar("T")


def merge_sort(arr: List[T]) -> List[T]:
    """
    Sort a list using the merge sort algorithm.

    This function returns a new sorted list and does not modify the original.

    Time Complexity:  O(n log n) in the best, average, and worst cases
    Space Complexity: O(n) for the auxiliary lists created during merging

    Merge sort is stable: elements with equal keys retain their original order.

    Args:
        arr: The list of comparable elements to sort.

    Returns:
        A new list containing the elements of `arr` in ascending order.
    """
    # Base case: lists with 0 or 1 element are already sorted.
    if len(arr) <= 1:
        return arr[:]  # return a copy to keep behavior consistent

    # Divide: split the list into two halves.
    mid = len(arr) // 2
    left_half = merge_sort(arr[:mid])
    right_half = merge_sort(arr[mid:])

    # Conquer: merge the two sorted halves.
    return _merge(left_half, right_half)


def _merge(left: List[T], right: List[T]) -> List[T]:
    """
    Merge two sorted lists into one sorted list.

    Uses '<=' when comparing so that equal elements from the left list
    are placed first, preserving stability.
    """
    merged: List[T] = []
    i = j = 0

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    # Append any remaining elements from either list.
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


def merge_sort_in_place(arr: List[T], low: int = 0, high: int | None = None) -> None:
    """
    Sort a list in place using merge sort.

    Uses a temporary buffer internally, so auxiliary space is O(n).

    Args:
        arr: The list to sort (modified in place).
        low: Starting index of the segment to sort (inclusive).
        high: Ending index of the segment to sort (exclusive). Defaults to len(arr).
    """
    if high is None:
        high = len(arr)

    if high - low <= 1:
        return

    mid = (low + high) // 2
    merge_sort_in_place(arr, low, mid)
    merge_sort_in_place(arr, mid, high)
    _merge_in_place(arr, low, mid, high)


def _merge_in_place(arr: List[T], low: int, mid: int, high: int) -> None:
    """Merge the sorted subarrays arr[low:mid] and arr[mid:high] in place."""
    buffer = arr[low:high]  # temporary copy of the segment
    i, j, k = 0, mid - low, low
    left_end = mid - low
    right_end = high - low

    while i < left_end and j < right_end:
        if buffer[i] <= buffer[j]:
            arr[k] = buffer[i]
            i += 1
        else:
            arr[k] = buffer[j]
            j += 1
        k += 1

    while i < left_end:
        arr[k] = buffer[i]
        i += 1
        k += 1

    while j < right_end:
        arr[k] = buffer[j]
        j += 1
        k += 1


if __name__ == "__main__":
    # Example usage
    data = [38, 27, 43, 3, 9, 82, 10]
    print("Original:        ", data)
    print("merge_sort:      ", merge_sort(data))
    print("Original (after):", data)  # unchanged

    merge_sort_in_place(data)
    print("In-place sorted: ", data)

    # Works with any comparable type
    words = ["banana", "apple", "cherry", "date"]
    print("Strings:         ", merge_sort(words))

    # Edge cases
    print("Empty:           ", merge_sort([]))
    print("Single element:  ", merge_sort([42]))
    print("Duplicates:      ", merge_sort([5, 2, 5, 1, 2]))
```

**Expected output:**
```
Original:         [38, 27, 43, 3, 9, 82, 10]
merge_sort:       [3, 9, 10, 27, 38, 43, 82]
Original (after): [38, 27, 43, 3, 9, 82, 10]
In-place sorted:  [3, 9, 10, 27, 38, 43, 82]
Strings:          ['apple', 'banana', 'cherry', 'date']
Empty:            []
Single element:   [42]
Duplicates:       [1, 2, 2, 5, 5]
```

**Key points:**

- **Divide and conquer:** The list is recursively split in half until each piece has one element, then the pieces are merged back in sorted order.
- **Stability:** Using `<=` in the merge step keeps equal elements in their original relative order.
- **Two versions:** `merge_sort` returns a new list (simpler, O(n) extra space), while `merge_sort_in_place` sorts the existing list using a temporary buffer for merging.
- **Complexity:** O(n log n) time in all cases, since the list is always split into halves regardless of its initial order.

## Step 2

As you can see the merge sort output looks really good and already includes an output trace so we shouldn't expect much issues/feedback. The next step is the pass that output back into the LLM and ask it to suggest fixes to any issues it finds within the code

In [ ]:
reflection_system = "You are Andrej Karpathy, an experienced computer scientist. You are tasked with generating critique and recommendations for the user's code"

reflection_user = [{'role': 'user', 'content': resp}]

refl_r = client.messages.create(
    max_tokens = 8000,
    system=reflection_system,
    messages=reflection_user,
    model=MODEL
)

refl_resp = "".join(b.text for b in refl_r.content if b.type == "text")



In [8]:
display_markdown(refl_resp, raw=True)

## Review: Merge Sort Implementation

The algorithm is correct. I traced the recursion and both merge routines by hand, and the expected outputs match what the code produces. The structure is clear, and the docstrings state the complexity and stability guarantees. The issues below are about API design, portability, and testing, not correctness.

### 1. Stability depends on `<=`, which is more than the sort needs

The code uses `left[i] <= right[j]`, so stability requires `__le__` on the element type. Python's sorting protocol only requires `__lt__`. A class that defines only `__lt__` (common with `functools.total_ordering` omitted, or with minimal domain types) will raise `TypeError` here even though `sorted()` works fine.

The stable version only needs `<`: take from the right only when it is strictly smaller.

```python
def _merge(left: List[T], right: List[T]) -> List[T]:
    merged: List[T] = []
    i = j = 0
    while i < len(left) and j < len(right):
        if right[j] < left[i]:      # strict: ties keep the left element first
            merged.append(right[j])
            j += 1
        else:
            merged.append(left[i])
            i += 1
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged
```

### 2. The two merge routines duplicate logic

`_merge` and `_merge_in_place` implement the same algorithm with different memory layouts. Maintaining two copies invites divergence, and you already have a bug-prone pair to keep in sync. The out-of-place version can be a thin wrapper:

```python
def merge_sort(arr: List[T]) -> List[T]:
    result = list(arr)
    merge_sort_in_place(result)
    return result
```

This removes the per-level slicing (`arr[:mid]`, `arr[mid:]`), which allocates O(n) elements at every recursion level. That is still O(n log n) total allocation, and it adds constant-factor overhead that the in-place version avoids. Your docstring's "O(n) for auxiliary lists" is also slightly misleading. The live memory is O(n), but the code churns through O(n log n) total allocations.

### 3. Python version compatibility

`high: int | None` in a signature is evaluated at definition time, so it requires Python 3.10+. If you want to support older versions, add `from __future__ import annotations` or use `Optional[int]` from `typing`, which you already import for the other hints.

### 4. No `key` parameter, so stability is untestable from outside

Stability is a headline property, but the API offers no way to demonstrate it. Sorting plain integers or strings cannot reveal instability, because equal elements are indistinguishable. Adding a `key` argument, as `sorted()` does, makes stability verifiable:

```python
def merge_sort(arr, key=None):
    ...
```

Then a test can sort records by one field and check that the other field's original order is preserved for ties.

### 5. Naming and docs

- `merge_sort_in_place` is an "in-place interface with O(n) scratch space." That is the standard meaning of the term in practice, but say so explicitly in the docstring, since "in place" sometimes implies O(1) extra space.
- The expected-output block has inconsistent column alignment relative to the `print` calls. Minor, but if someone copies the output into a doc test, mismatches will cause confusion. Consider using `doctest` so the examples are verified automatically.

### 6. Missing tests

The `__main__` block shows a few examples, but it does not check correctness. Add a randomized comparison against the built-in:

```python
import random

def test_matches_builtin():
    for _ in range(1000):
        xs = [random.randint(-50, 50) for _ in range(random.randint(0, 200))]
        assert merge_sort(xs) == sorted(xs)
        ys = xs[:]
        merge_sort_in_place(ys)
        assert ys == sorted(xs)

def test_stability():
    # Requires the key parameter from point 4
    records = [(1, "a"), (0, "b"), (1, "c"), (0, "d")]
    out = merge_sort(records, key=lambda r: r[0])
    assert out == [(0, "b"), (0, "d"), (1, "a"), (1, "c")]
```

Randomized testing against `sorted()` will catch off-by-one errors in the index arithmetic far more reliably than hand-picked examples. The in-place version's boundary handling (`low`, `high`, `mid`) is the most error-prone part, and it is currently covered only by the single example.

### Summary

| Area | Status |
|---|---|
| Algorithm correctness | Correct |
| Stability | Correct, but relies on `<=` where `<` suffices |
| Code duplication | Two merge routines; collapse to one |
| Compatibility | `int \| None` requires Python 3.10+ |
| API | Add `key` to make stability testable |
| Testing | Add randomized comparison against `sorted()` |

The core is solid. The highest-value changes are switching to `<` (so the sort works with any type that supports `sorted()`), removing the duplicated merge logic, and adding a randomized test. I'd do those three before anything else.

## The loop

Now we simply put everything together by feeding the refinements back into the model and allow it to update it's results. This is done in a simple loop. There are two main termination types the first being there are no more refinements where you tell the refinement LLM to output "<OK>" if there is nothing to refine. The second is if you reach max steps which is important because without tracking the steps you rist running an infinite loop which eats up an extreme amount of tokens

In [3]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.patterns import ReflectionAgent

In [4]:
agent = ReflectionAgent(client=client)

reflection_results = agent.run(user_msg="Generate a Python implementation of the Merge Sort Algorithm")

## Reflection Loop Stats

In [10]:
print(f'Total Steps: {reflection_results.steps}\n')
print(f'Total Input Tokens: {reflection_results.input_tokens}\n')
print(f'Total Output Tokens: {reflection_results.output_tokens}\n')
print(f'Total Latency: {reflection_results.latency}')

Total Steps: 2

Total Input Tokens: 7313

Total Output Tokens: 7217

Total Latency: 31.10043800016865


## Original Output

In [8]:
display_markdown(reflection_results.revisions[0], raw=True)

```python
def merge_sort(arr):
    """
    Sort a list using the merge sort algorithm.
    Returns a new sorted list; the original list is not modified.
    Time complexity: O(n log n) in all cases.
    Space complexity: O(n) for the auxiliary arrays.
    """
    # Base case: a list of 0 or 1 elements is already sorted
    if len(arr) <= 1:
        return arr[:]

    # Divide: split the list into two halves
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    # Conquer: merge the two sorted halves
    return merge(left, right)


def merge(left, right):
    """
    Merge two sorted lists into a single sorted list.
    """
    merged = []
    i = j = 0

    # Compare elements from both lists and take the smaller one
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:  # <= keeps the sort stable
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    # Append any remaining elements from either list
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


def merge_sort_in_place(arr, lo=0, hi=None):
    """
    In-place version of merge sort that sorts arr[lo:hi] directly.
    """
    if hi is None:
        hi = len(arr)

    if hi - lo <= 1:
        return arr

    mid = (lo + hi) // 2
    merge_sort_in_place(arr, lo, mid)
    merge_sort_in_place(arr, mid, hi)
    _merge_in_place(arr, lo, mid, hi)
    return arr


def _merge_in_place(arr, lo, mid, hi):
    """
    Merge the sorted subarrays arr[lo:mid] and arr[mid:hi] in place.
    """
    left = arr[lo:mid]
    right = arr[mid:hi]
    i = j = 0
    k = lo

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            arr[k] = left[i]
            i += 1
        else:
            arr[k] = right[j]
            j += 1
        k += 1

    while i < len(left):
        arr[k] = left[i]
        i += 1
        k += 1

    while j < len(right):
        arr[k] = right[j]
        j += 1
        k += 1


if __name__ == "__main__":
    data = [38, 27, 43, 3, 9, 82, 10]
    print("Original:          ", data)
    print("merge_sort:        ", merge_sort(data))

    data2 = [38, 27, 43, 3, 9, 82, 10]
    merge_sort_in_place(data2)
    print("merge_sort_in_place:", data2)

    # Edge cases
    print(merge_sort([]))             # []
    print(merge_sort([1]))            # [1]
    print(merge_sort([5, 5, 5, 2]))   # [2, 5, 5, 5]
    print(merge_sort([-3, 0, -1, 8])) # [-3, -1, 0, 8]
```

**How it works:**

1. **Divide:** The list is recursively split into halves until each piece has one element (which is trivially sorted).
2. **Conquer:** Adjacent sorted pieces are merged by repeatedly taking the smaller front element from the two lists.
3. **Result:** Merging continues up the recursion tree until the whole list is sorted.

**Complexity:**
- Time: O(n log n) in best, average, and worst cases
- Space: O(n) auxiliary space (the in-place variant still uses O(n) temporary space during each merge)

**Output:**
```
Original:           [38, 27, 43, 3, 9, 82, 10]
merge_sort:         [3, 9, 10, 27, 38, 43, 82]
merge_sort_in_place: [3, 9, 10, 27, 38, 43, 82]
[]
[1]
[2, 5, 5, 5]
[-3, -1, 0, 8]
```

## Final Output

In [9]:
display_markdown(reflection_results.output, raw=True)

```python
def merge_sort(arr):
    """
    Sort a list using the merge sort algorithm.

    Returns a new sorted list; the input list is not modified.

    Time complexity:  O(n log n) in the best, average, and worst cases.
    Space complexity: O(n) auxiliary space for the merged lists and slices,
                      plus O(log n) recursion stack depth. Slicing copies
                      O(n) elements per level, so the total copying work
                      is O(n log n).
    """
    # Base case: a list of 0 or 1 elements is already sorted.
    # Return a copy rather than the original object so every call returns
    # a new list, consistent with the "returns a new list" contract.
    if len(arr) <= 1:
        return arr[:]

    # Divide: split the list into two halves.
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    # Conquer: merge the two sorted halves.
    return merge(left, right)


def merge(left, right):
    """
    Merge two sorted lists into a single sorted list.
    """
    merged = []
    i = j = 0

    # Take the smaller front element from the two lists.
    # The <= comparison takes from the left list on ties, which keeps the sort stable.
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    # One list is exhausted; append the remainder of the other.
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


def merge_sort_inplace_buffered(arr, lo=0, hi=None):
    """
    Sort arr[lo:hi] in place using merge sort.

    The sort itself writes results back into arr, but each merge step
    copies both halves into temporary lists. This is NOT O(1) extra space:
    it uses O(n) temporary space during merges, plus O(log n) recursion stack.

    Time complexity:  O(n log n) in all cases.
    Space complexity: O(n) temporary buffers, O(log n) recursion stack.
    """
    if hi is None:
        hi = len(arr)

    if hi - lo <= 1:
        return arr

    mid = (lo + hi) // 2
    merge_sort_inplace_buffered(arr, lo, mid)
    merge_sort_inplace_buffered(arr, mid, hi)
    _merge_inplace_buffered(arr, lo, mid, hi)
    return arr


def _merge_inplace_buffered(arr, lo, mid, hi):
    """
    Merge the sorted subarrays arr[lo:mid] and arr[mid:hi] back into arr.
    Uses O(hi - lo) temporary space for copies of both halves.
    """
    left = arr[lo:mid]
    right = arr[mid:hi]
    i = j = 0
    k = lo

    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            arr[k] = left[i]
            i += 1
        else:
            arr[k] = right[j]
            j += 1
        k += 1

    while i < len(left):
        arr[k] = left[i]
        i += 1
        k += 1

    while j < len(right):
        arr[k] = right[j]
        j += 1
        k += 1


def show(label, value, width=30):
    """Print a label and value with the value aligned in a fixed column."""
    print(f"{label:<{width}}{value}")


if __name__ == "__main__":
    data = [38, 27, 43, 3, 9, 82, 10]
    show("Original:", data)
    show("merge_sort:", merge_sort(data))

    data2 = [38, 27, 43, 3, 9, 82, 10]
    merge_sort_inplace_buffered(data2)
    show("merge_sort_inplace_buffered:", data2)

    print()
    print("Edge cases:")
    for case in ([], [1], [5, 5, 5, 2], [-3, 0, -1, 8]):
        show(f"merge_sort({case}):", merge_sort(case))
```

**How it works:**

1. **Divide:** The list is recursively split into halves until each piece has one element, which is trivially sorted.
2. **Conquer:** Adjacent sorted pieces are merged by repeatedly taking the smaller front element from the two lists.
3. **Result:** Merging continues up the recursion tree until the whole list is sorted.

**Complexity:**
- `merge_sort`: O(n log n) time. O(n) auxiliary space, plus O(log n) recursion stack.
- `merge_sort_inplace_buffered`: O(n log n) time. It writes back into the input list, but uses O(n) temporary buffers during each merge, plus O(log n) recursion stack. It is not O(1) extra space. A true O(1)-space in-place merge sort is considerably more involved and is not implemented here.

**Output:**
```
Original:                     [38, 27, 43, 3, 9, 82, 10]
merge_sort:                   [3, 9, 10, 27, 38, 43, 82]
merge_sort_inplace_buffered:  [3, 9, 10, 27, 38, 43, 82]

Edge cases:
merge_sort([]):               []
merge_sort([1]):              [1]
merge_sort([5, 5, 5, 2]):     [2, 5, 5, 5]
merge_sort([-3, 0, -1, 8]):   [-3, -1, 0, 8]
```